<a href="https://colab.research.google.com/github/karthik14344/Machine_learning/blob/Regression/electronics_store.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import pandas as pd
import numpy as np
import ast
import warnings
from math import radians, sin, cos, sqrt, atan2, ceil, log
from scipy import stats
from scipy.sparse import hstack as sparse_hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.metrics import roc_auc_score, mean_absolute_error
from sklearn.feature_selection import mutual_info_classif
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF
from sklearn.ensemble import RandomForestRegressor
from sklearn.decomposition import NMF
import statsmodels.api as sm
import plotly.graph_objects as go
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

warnings.filterwarnings('ignore')

import os
BASE = "./outputs"
os.makedirs(BASE, exist_ok=True)


# ============================================================
# PART 1: DATA LOADING AND CLEANING
# ============================================================

df_dirty = pd.read_excel('dirty_data.xlsx')
df_missing = pd.read_excel('missing_data.xlsx')
df_wh = pd.read_excel('warehouses.xlsx')


# --- Clean dirty_data ---
df_dirty['nearest_warehouse'] = df_dirty['nearest_warehouse'].str.title()
df_dirty['season'] = df_dirty['season'].str.title()
df_dirty['date'] = pd.to_datetime(df_dirty['date'])

# --- Clean missing_data ---
df_missing['date'] = pd.to_datetime(df_missing['date'])

# Impute season from date (Southern Hemisphere meteorological)
def get_season_sh(dt):
    m = dt.month
    if m in [12, 1, 2]:
        return 'Summer'
    elif m in [3, 4, 5]:
        return 'Autumn'
    elif m in [6, 7, 8]:
        return 'Winter'
    else:
        return 'Spring'

mask_season = df_missing['season'].isnull()
df_missing.loc[mask_season, 'season'] = df_missing.loc[mask_season, 'date'].apply(get_season_sh)

# Haversine function
def haversine(lat1, lon1, lat2, lon2, R=6371):
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = sin(dlat/2)**2 + cos(lat1)*cos(lat2)*sin(dlon/2)**2
    return R * 2 * atan2(sqrt(a), sqrt(1-a))

# Drop rows missing customer_lat (10 rows) — can't locate without latitude
df_missing = df_missing.dropna(subset=['customer_lat']).copy()

# Impute nearest_warehouse for missing rows using haversine to all three warehouses
wh_coords = df_wh.set_index('names')[['lat', 'lon']].to_dict('index')
mask_wh = df_missing['nearest_warehouse'].isnull()
for idx in df_missing[mask_wh].index:
    clat = df_missing.loc[idx, 'customer_lat']
    clon = df_missing.loc[idx, 'customer_long']
    best_wh = min(wh_coords.keys(), key=lambda w: haversine(clat, clon, wh_coords[w]['lat'], wh_coords[w]['lon']))
    df_missing.loc[idx, 'nearest_warehouse'] = best_wh

# Fill missing numerics with median grouped by nearest_warehouse
numeric_cols_to_fill = ['order_price', 'order_total', 'distance_to_nearest_warehouse', 'customer_long']
for col in numeric_cols_to_fill:
    df_missing[col] = df_missing.groupby('nearest_warehouse')[col].transform(
        lambda x: x.fillna(x.median())
    )

# Drop rows missing is_happy_customer (10 rows)
df_missing = df_missing.dropna(subset=['is_happy_customer']).copy()
print(f"missing_data cleaned shape: {df_missing.shape}")

# Cast is_happy_customer to bool
df_missing['is_happy_customer'] = df_missing['is_happy_customer'].astype(bool)

# --- Merge both with warehouses ---
df_dirty_m = df_dirty.merge(df_wh, left_on='nearest_warehouse', right_on='names', how='left', suffixes=('', '_wh'))
df_missing_m = df_missing.merge(df_wh, left_on='nearest_warehouse', right_on='names', how='left', suffixes=('', '_wh'))

print("Dirty merged shape:", df_dirty_m.shape)
print("Missing merged shape:", df_missing_m.shape)

# ============================================================
# PART 2: SHOPPING CART PARSING, ENTROPY, HAVERSINE VALIDATION
# ============================================================

def parse_cart(cart_str):
    try:
        return ast.literal_eval(cart_str)
    except:
        return []

df_dirty_m['parsed_cart'] = df_dirty_m['shopping_cart'].apply(parse_cart)
df_dirty_m['cart_size'] = df_dirty_m['parsed_cart'].apply(lambda c: len(set(item[0] for item in c)))
df_dirty_m['total_quantity'] = df_dirty_m['parsed_cart'].apply(lambda c: sum(item[1] for item in c))

# Explode for product frequency
all_products = []
for cart in df_dirty_m['parsed_cart']:
    for product, qty in cart:
        all_products.extend([product] * qty)

product_counts = pd.Series(all_products).value_counts()
probs = product_counts / product_counts.sum()
shannon_entropy = -np.sum(probs * np.log(probs))
print(f"\nShannon Entropy (nats): {round(shannon_entropy, 4)}")

# Haversine validation
df_dirty_m['computed_distance'] = df_dirty_m.apply(
    lambda r: haversine(r['customer_lat'], r['customer_long'], r['lat'], r['lon']), axis=1
)
mean_abs_discrepancy = np.mean(np.abs(df_dirty_m['computed_distance'] - df_dirty_m['distance_to_nearest_warehouse']))
print(f"Mean Absolute Discrepancy (km): {round(mean_abs_discrepancy, 4)}")

# ============================================================
# PART 3: QUANTILE REGRESSION
# ============================================================

season_dummies_dirty = pd.get_dummies(df_dirty_m['season'], drop_first=False)
season_dummies_dirty = season_dummies_dirty.drop('Autumn', axis=1)

X_qr = pd.concat([
    df_dirty_m[['distance_to_nearest_warehouse', 'is_expedited_delivery']].astype(float),
    season_dummies_dirty.astype(float)
], axis=1)
X_qr = sm.add_constant(X_qr)
y_qr = df_dirty_m['delivery_charges'].astype(float)

mod_50 = sm.QuantReg(y_qr, X_qr).fit(q=0.5, max_iter=10000)
mod_90 = sm.QuantReg(y_qr, X_qr).fit(q=0.9, max_iter=10000)

dist_coef_50 = mod_50.params['distance_to_nearest_warehouse']
dist_coef_90 = mod_90.params['distance_to_nearest_warehouse']
print(f"\nQuantile Regression distance coef at tau=0.5: {round(dist_coef_50, 4)}")
print(f"Quantile Regression distance coef at tau=0.9: {round(dist_coef_90, 4)}")

# ============================================================
# PART 4: IPW PROPENSITY SCORE ATT + BOOTSTRAP
# ============================================================

season_dummies_ipw = pd.get_dummies(df_dirty_m['season'], drop_first=False).drop('Autumn', axis=1).astype(float)

X_ps = pd.concat([
    df_dirty_m[['distance_to_nearest_warehouse', 'delivery_charges', 'coupon_discount',
                 'cart_size', 'total_quantity']].astype(float),
    season_dummies_ipw
], axis=1)

treatment = df_dirty_m['is_expedited_delivery'].astype(int).values
outcome = df_dirty_m['is_happy_customer'].astype(int).values

ps_model = LogisticRegression(solver='lbfgs', max_iter=1000)
ps_model.fit(X_ps, treatment)
ps = ps_model.predict_proba(X_ps)[:, 1]

# Clip propensity scores
ps_clipped = np.clip(ps, np.percentile(ps, 1), np.percentile(ps, 99))

def compute_att(treatment, outcome, ps):
    w1 = treatment.copy().astype(float)
    w0 = ((1 - treatment) * ps / (1 - ps)).astype(float)
    att = np.sum(w1 * outcome) / np.sum(w1) - np.sum(w0 * outcome) / np.sum(w0)
    return att

att_full = compute_att(treatment, outcome, ps_clipped)
print(f"\nIPW ATT: {round(att_full, 4)}")

# Bootstrap 1000 times
rng = np.random.RandomState(42)
att_boots = []
n = len(treatment)
for _ in range(1000):
    idx = rng.choice(n, size=n, replace=True)
    att_b = compute_att(treatment[idx], outcome[idx], ps_clipped[idx])
    att_boots.append(att_b)

ci_lower = np.percentile(att_boots, 2.5)
ci_upper = np.percentile(att_boots, 97.5)
print(f"ATT Bootstrap 95% CI: [{round(ci_lower, 4)}, {round(ci_upper, 4)}]")
print(f"ATT 95% CI Lower Bound: {round(ci_lower, 4)}")

# ============================================================
# PART 5: TF-IDF + LOGISTIC REGRESSION CV
# ============================================================

df_dirty_m['latest_customer_review'] = df_dirty_m['latest_customer_review'].fillna('')

tfidf = TfidfVectorizer(ngram_range=(1, 2), max_features=500, min_df=5)
X_tfidf = tfidf.fit_transform(df_dirty_m['latest_customer_review'])

X_struct = df_dirty_m[['distance_to_nearest_warehouse', 'delivery_charges', 'coupon_discount',
                        'cart_size', 'total_quantity', 'is_expedited_delivery']].astype(float).values

X_combined = sparse_hstack([X_tfidf, X_struct])
y_lr = df_dirty_m['is_happy_customer'].astype(int).values

lr_model = LogisticRegression(solver='lbfgs', max_iter=1000, random_state=42)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(lr_model, X_combined, y_lr, cv=skf, scoring='roc_auc')
mean_roc_auc = np.mean(cv_scores)
print(f"\nTF-IDF + Logistic Regression 5-Fold Stratified CV Mean ROC AUC: {round(mean_roc_auc, 4)}")

# ============================================================
# PART 6: MUTUAL INFORMATION ON MISSING_DATA
# ============================================================

X_mi = df_missing_m['is_expedited_delivery'].astype(int).values.reshape(-1, 1)
y_mi = df_missing_m['is_happy_customer'].astype(bool).astype(int).values

mi_val = mutual_info_classif(X_mi, y_mi, discrete_features=True, random_state=42)[0]
print(f"\nMutual Information (is_expedited_delivery, is_happy_customer): {round(mi_val, 4)}")

# ============================================================
# PART 7: MONTE CARLO SIMULATION
# ============================================================

rng_mc = np.random.RandomState(42)
p90 = df_dirty_m['order_total'].quantile(0.9)
n_sims = 10000
count = 0
for _ in range(n_sims):
    idx = rng_mc.randint(0, len(df_dirty_m))
    row = df_dirty_m.iloc[idx]
    if row['order_total'] > p90 and row['is_happy_customer']:
        count += 1

joint_prob = count / n_sims
print(f"\nMonte Carlo Joint Probability (order_total > P90 AND happy): {round(joint_prob, 4)}")

# ============================================================
# PART 8: GAUSSIAN PROCESS REGRESSION
# ============================================================

X_gp = df_dirty_m[['distance_to_nearest_warehouse']].values
y_gp = df_dirty_m['delivery_charges'].values

kernel = RBF(length_scale=1.0)
gpr = GaussianProcessRegressor(kernel=kernel, alpha=1.0, random_state=42)
gpr.fit(X_gp, y_gp)
y_gp_pred = gpr.predict(X_gp)
gp_mae = mean_absolute_error(y_gp, y_gp_pred)
print(f"\nGP Regression MAE (train): {round(gp_mae, 4)}")

# ============================================================
# PART 9: McNEMAR'S TEST
# ============================================================

# Parse cart for missing_data too
df_missing_m['parsed_cart'] = df_missing_m['shopping_cart'].apply(parse_cart)
df_missing_m['cart_size'] = df_missing_m['parsed_cart'].apply(lambda c: len(set(item[0] for item in c)))
df_missing_m['total_quantity'] = df_missing_m['parsed_cart'].apply(lambda c: sum(item[1] for item in c))

features_mcn = ['distance_to_nearest_warehouse', 'delivery_charges', 'coupon_discount', 'is_expedited_delivery']

# Train on dirty
X_d = df_dirty_m[features_mcn].astype(float)
y_d = df_dirty_m['is_happy_customer'].astype(int)
lr_dirty = LogisticRegression(solver='lbfgs', max_iter=1000, random_state=42)
lr_dirty.fit(X_d, y_d)
pred_dirty = lr_dirty.predict(X_d)

# Train on missing
X_m = df_missing_m[features_mcn].astype(float)
y_m = df_missing_m['is_happy_customer'].astype(int)
lr_missing = LogisticRegression(solver='lbfgs', max_iter=1000, random_state=42)
lr_missing.fit(X_m, y_m)
pred_missing = lr_missing.predict(X_m)

# First 100 predictions from each
pred_d_100 = pred_dirty[:100]
pred_m_100 = pred_missing[:100]

# McNemar's test (no continuity correction)
b = np.sum((pred_d_100 == 1) & (pred_m_100 == 0))
c = np.sum((pred_d_100 == 0) & (pred_m_100 == 1))
mcnemar_stat = (b - c)**2 / (b + c) if (b + c) > 0 else 0.0
print(f"\nMcNemar's Test Statistic: {round(mcnemar_stat, 4)}")

# ============================================================
# PART 10: MEDIATION ANALYSIS
# ============================================================

# Mediator model: delivery_charges ~ distance (OLS)
X_med_a = sm.add_constant(df_dirty_m['distance_to_nearest_warehouse'].astype(float))
y_med_a = df_dirty_m['delivery_charges'].astype(float)
med_a_model = sm.OLS(y_med_a, X_med_a).fit()
a_coef = med_a_model.params['distance_to_nearest_warehouse']

# Outcome model: is_happy_customer ~ distance + delivery_charges (logistic)
X_med_b = sm.add_constant(df_dirty_m[['distance_to_nearest_warehouse', 'delivery_charges']].astype(float))
y_med_b = df_dirty_m['is_happy_customer'].astype(int)
med_b_model = sm.Logit(y_med_b, X_med_b).fit(disp=0)
b_coef = med_b_model.params['delivery_charges']

indirect_effect = a_coef * b_coef
print(f"\nMediation Indirect Effect (a*b): {round(indirect_effect, 4)}")

# ============================================================
# PART 11: CONFORMAL PREDICTION
# ============================================================

X_conf = df_dirty_m[['distance_to_nearest_warehouse', 'is_expedited_delivery', 'coupon_discount', 'cart_size']].astype(float)
y_conf = df_dirty_m['delivery_charges'].astype(float)

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(X_conf, y_conf, test_size=0.2, random_state=42)

rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train_c, y_train_c)

# Nonconformity scores on calibration set (training set used as calibration)
y_train_pred = rf.predict(X_train_c)
residuals = np.abs(y_train_c.values - y_train_pred)

n_cal = len(residuals)
q_level = ceil((n_cal + 1) * 0.9) / n_cal
q_val = np.quantile(residuals, q_level)

y_test_pred = rf.predict(X_test_c)
lower = y_test_pred - q_val
upper = y_test_pred + q_val
mean_interval_width = np.mean(upper - lower)
print(f"\nConformal Prediction Mean Interval Width: {round(mean_interval_width, 4)}")

# ============================================================
# PART 12: NMF
# ============================================================

# Build binary order-by-product matrix
all_product_names = set()
for cart in df_dirty_m['parsed_cart']:
    for product, qty in cart:
        all_product_names.add(product)
all_product_names = sorted(all_product_names)

product_to_idx = {p: i for i, p in enumerate(all_product_names)}
n_orders = len(df_dirty_m)
n_products = len(all_product_names)
order_product_matrix = np.zeros((n_orders, n_products))

for i, cart in enumerate(df_dirty_m['parsed_cart']):
    for product, qty in cart:
        order_product_matrix[i, product_to_idx[product]] = 1

nmf_model = NMF(n_components=3, random_state=42, max_iter=500)
W = nmf_model.fit_transform(order_product_matrix)
H = nmf_model.components_
reconstructed = W @ H
frob_error = np.linalg.norm(order_product_matrix - reconstructed, 'fro')
print(f"\nNMF Frobenius Norm Reconstruction Error: {round(frob_error, 4)}")

# ============================================================
# PART 13: ATT BY DISTANCE QUARTILES (for forest plot)
# ============================================================

df_dirty_m['dist_quartile'] = pd.qcut(df_dirty_m['distance_to_nearest_warehouse'], q=4, duplicates='drop')
quartile_labels = df_dirty_m['dist_quartile'].cat.categories

att_by_quartile = {}
ci_by_quartile = {}

for q_label in quartile_labels:
    mask = df_dirty_m['dist_quartile'] == q_label
    t_q = treatment[mask]
    o_q = outcome[mask]
    ps_q = ps_clipped[mask]

    att_q = compute_att(t_q, o_q, ps_q)
    att_by_quartile[str(q_label)] = att_q

    rng_q = np.random.RandomState(42)
    boots_q = []
    n_q = len(t_q)
    for _ in range(1000):
        idx_q = rng_q.choice(n_q, size=n_q, replace=True)
        boots_q.append(compute_att(t_q[idx_q], o_q[idx_q], ps_q[idx_q]))
    ci_by_quartile[str(q_label)] = (np.percentile(boots_q, 2.5), np.percentile(boots_q, 97.5))

print("\nATT by Distance Quartile:")
for q, att_val in att_by_quartile.items():
    ci = ci_by_quartile[q]
    print(f"  {q}: ATT={round(att_val, 4)}, 95% CI=[{round(ci[0], 4)}, {round(ci[1], 4)}]")

# ============================================================
# PART 14: PLOTS
# ============================================================

# --- PLOT 1: Sankey Diagram ---
sankey_data = df_dirty_m.groupby(['nearest_warehouse', 'season', 'is_happy_customer']).size().reset_index(name='count')

warehouses_list = sorted(df_dirty_m['nearest_warehouse'].unique())
seasons_list = sorted(df_dirty_m['season'].unique())
happy_list = [False, True]

labels = warehouses_list + seasons_list + ['Unhappy', 'Happy']

wh_to_idx = {w: i for i, w in enumerate(warehouses_list)}
season_to_idx = {s: i + len(warehouses_list) for i, s in enumerate(seasons_list)}
happy_to_idx = {False: len(warehouses_list) + len(seasons_list), True: len(warehouses_list) + len(seasons_list) + 1}

sources = []
targets = []
values = []

# Warehouse -> Season
wh_season = df_dirty_m.groupby(['nearest_warehouse', 'season']).size().reset_index(name='count')
for _, row in wh_season.iterrows():
    sources.append(wh_to_idx[row['nearest_warehouse']])
    targets.append(season_to_idx[row['season']])
    values.append(row['count'])

# Season -> Happy/Unhappy
for _, row in sankey_data.iterrows():
    sources.append(season_to_idx[row['season']])
    targets.append(happy_to_idx[row['is_happy_customer']])
    values.append(row['count'])

fig_sankey = go.Figure(data=[go.Sankey(
    node=dict(pad=15, thickness=20, line=dict(color="black", width=0.5), label=labels),
    link=dict(source=sources, target=targets, value=values)
)])
fig_sankey.update_layout(title_text="Warehouse → Season → Customer Happiness", font_size=12)
fig_sankey.write_html(f'{BASE}/sankey.html')
print("\nSankey diagram saved to sankey.html")

# --- PLOT 2: Forest Plot of ATT by Distance Quartile ---
fig_forest, ax_forest = plt.subplots(figsize=(10, 6))
q_labels_list = list(att_by_quartile.keys())
att_vals = [att_by_quartile[q] for q in q_labels_list]
ci_lowers = [ci_by_quartile[q][0] for q in q_labels_list]
ci_uppers = [ci_by_quartile[q][1] for q in q_labels_list]
errors = [[att_vals[i] - ci_lowers[i] for i in range(len(q_labels_list))],
          [ci_uppers[i] - att_vals[i] for i in range(len(q_labels_list))]]

y_pos = range(len(q_labels_list))
ax_forest.errorbar(att_vals, y_pos, xerr=errors, fmt='o', color='steelblue', capsize=5, markersize=8)
ax_forest.axvline(x=0, color='red', linestyle='--', linewidth=1)
ax_forest.set_yticks(list(y_pos))
ax_forest.set_yticklabels(q_labels_list)
ax_forest.set_xlabel('ATT Estimate')
ax_forest.set_title('Forest Plot: ATT by Distance Quartile (with 95% Bootstrap CI)')
plt.tight_layout()
plt.savefig(f'{BASE}/forest_plot.png', dpi=150)
plt.close()
print("Forest plot saved to forest_plot.png")

# --- PLOT 3: CUSUM Chart ---
df_dirty_m['date_only'] = df_dirty_m['date'].dt.date
daily_mean = df_dirty_m.groupby('date_only')['delivery_charges'].mean()
grand_mean = df_dirty_m['delivery_charges'].mean()
deviations = daily_mean - grand_mean
cusum = deviations.cumsum()
std_dev = deviations.std()
threshold = 2 * std_dev

fig_cusum, ax_cusum = plt.subplots(figsize=(14, 6))
dates = pd.to_datetime(cusum.index)
ax_cusum.plot(dates, cusum.values, color='steelblue', linewidth=1.5, marker='.', markersize=3)
ax_cusum.axhline(y=threshold, color='orange', linestyle='--', label=f'+2σ = {round(threshold, 2)}')
ax_cusum.axhline(y=-threshold, color='orange', linestyle='--', label=f'-2σ = {round(-threshold, 2)}')

# Flag points exceeding 2 std devs
exceed_mask = np.abs(cusum.values) > threshold
if np.any(exceed_mask):
    ax_cusum.scatter(dates[exceed_mask], cusum.values[exceed_mask], color='red', s=50, zorder=5, label='Exceeds 2σ')

ax_cusum.set_xlabel('Date')
ax_cusum.set_ylabel('CUSUM')
ax_cusum.set_title('CUSUM Chart of Daily Mean Delivery Charges Deviations')
ax_cusum.legend()
plt.tight_layout()
plt.savefig(f'{BASE}/cusum_chart.png', dpi=150)
plt.close()
print("CUSUM chart saved to cusum_chart.png")

# --- PLOT 4: Connected Scatterplot ---
df_dirty_m['year_month'] = df_dirty_m['date'].dt.to_period('M')
monthly = df_dirty_m.groupby('year_month').agg(
    avg_distance=('distance_to_nearest_warehouse', 'mean'),
    avg_delivery=('delivery_charges', 'mean')
).reset_index()
monthly['year_month_str'] = monthly['year_month'].astype(str)

fig_scatter, ax_scatter = plt.subplots(figsize=(12, 8))
ax_scatter.plot(monthly['avg_distance'], monthly['avg_delivery'], '-o', color='steelblue', markersize=6)

for i in range(len(monthly)):
    ax_scatter.annotate(monthly['year_month_str'].iloc[i],
                        (monthly['avg_distance'].iloc[i], monthly['avg_delivery'].iloc[i]),
                        textcoords="offset points", xytext=(5, 5), fontsize=7)

# Arrows
for i in range(len(monthly) - 1):
    dx = monthly['avg_distance'].iloc[i+1] - monthly['avg_distance'].iloc[i]
    dy = monthly['avg_delivery'].iloc[i+1] - monthly['avg_delivery'].iloc[i]
    ax_scatter.annotate('', xy=(monthly['avg_distance'].iloc[i+1], monthly['avg_delivery'].iloc[i+1]),
                        xytext=(monthly['avg_distance'].iloc[i], monthly['avg_delivery'].iloc[i]),
                        arrowprops=dict(arrowstyle='->', color='gray', lw=1.2))

ax_scatter.set_xlabel('Average Distance to Nearest Warehouse (km)')
ax_scatter.set_ylabel('Average Delivery Charges')
ax_scatter.set_title('Connected Scatterplot: Monthly Avg Distance vs Avg Delivery Charges')
plt.tight_layout()
plt.savefig(f'{BASE}/connected_scatter.png', dpi=150)
plt.close()
print("Connected scatterplot saved to connected_scatter.png")

# ============================================================
# SUMMARY OF ALL RESULTS
# ============================================================

print("\n" + "="*60)
print("SUMMARY OF ALL RESULTS")
print("="*60)
print(f"Shannon Entropy (nats): {round(shannon_entropy, 4)}")
print(f"Mean Absolute Haversine Discrepancy (km): {round(mean_abs_discrepancy, 4)}")
print(f"Quantile Reg distance coef tau=0.5: {round(dist_coef_50, 4)}")
print(f"Quantile Reg distance coef tau=0.9: {round(dist_coef_90, 4)}")
print(f"IPW ATT: {round(att_full, 4)}")
print(f"ATT 95% CI Lower Bound: {round(ci_lower, 4)}")
print(f"TF-IDF + LR Mean ROC AUC: {round(mean_roc_auc, 4)}")
print(f"Mutual Information (missing_data): {round(mi_val, 4)}")
print(f"Monte Carlo Joint Probability: {round(joint_prob, 4)}")
print(f"GP Regression MAE: {round(gp_mae, 4)}")
print(f"McNemar's Test Statistic: {round(mcnemar_stat, 4)}")
print(f"Mediation Indirect Effect (a*b): {round(indirect_effect, 4)}")
print(f"Conformal Prediction Mean Interval Width: {round(mean_interval_width, 4)}")
print(f"NMF Frobenius Norm Error: {round(frob_error, 4)}")


missing_data cleaned shape: (480, 16)
Dirty merged shape: (500, 19)
Missing merged shape: (480, 19)

Shannon Entropy (nats): 2.469
Mean Absolute Discrepancy (km): 221.1785

Quantile Regression distance coef at tau=0.5: -0.0107
Quantile Regression distance coef at tau=0.9: 0.0186

IPW ATT: -0.2643
ATT Bootstrap 95% CI: [-0.3277, -0.2038]
ATT 95% CI Lower Bound: -0.3277

TF-IDF + Logistic Regression 5-Fold Stratified CV Mean ROC AUC: 0.9027

Mutual Information (is_expedited_delivery, is_happy_customer): 0.0002

Monte Carlo Joint Probability (order_total > P90 AND happy): 0.0737

GP Regression MAE (train): 11.2281

McNemar's Test Statistic: 0.0

Mediation Indirect Effect (a*b): -0.0072

Conformal Prediction Mean Interval Width: 14.6976

NMF Frobenius Norm Reconstruction Error: 28.1136

ATT by Distance Quartile:
  (0.107, 0.751]: ATT=-0.2815, 95% CI=[-0.3944, -0.1641]
  (0.751, 1.03]: ATT=-0.2875, 95% CI=[-0.3991, -0.1737]
  (1.03, 1.409]: ATT=-0.2288, 95% CI=[-0.3674, -0.103]
  (1.409, 94